# 01 — Radar data audit
Status: unexecuted on real data. Documentation-driven starter, not experimental results.
Read `../docs/radar-data-audit.md` first. Confirm rights before using the binary file.
Requires NumPy, pandas, matplotlib, and a Jupyter kernel. Record the tested versions below.
Set RADAR_DATA_PATH to the official local download. Object-array deserialization uses pickle: enable only after verifying and trusting the official file.


In [ ]:
from pathlib import Path
import os, hashlib, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
print({'python': sys.version, 'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': matplotlib.__version__})
raw_path = os.environ.get('RADAR_DATA_PATH')
if not raw_path:
    raise RuntimeError('Set RADAR_DATA_PATH to the official downloaded .npy file.')
path = Path(raw_path).expanduser()
expected_md5 = '01d66ba7b1ccc04477a9e69b2813f251'
h = hashlib.md5()
with path.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        h.update(block)
if h.hexdigest() != expected_md5:
    raise ValueError('Checksum mismatch: stop and inspect the source/version.')
if os.environ.get('TRUST_OFFICIAL_RADAR_PICKLE') != 'yes':
    raise RuntimeError('After rights and source review, explicitly set TRUST_OFFICIAL_RADAR_PICKLE=yes.')
data = np.load(path, allow_pickle=True)
assert data.shape == (130, 6), f'Unexpected schema: {data.shape}'


In [ ]:
rows = []
for measurement_id, row in enumerate(data):
    label = str(np.asarray(row[0]).squeeze())
    signal = np.asarray(row[1])
    assert signal.ndim == 2 and signal.shape[0] == 1280, (measurement_id, signal.shape)
    assert np.iscomplexobj(signal), measurement_id
    n = signal.shape[1]
    ranges, times, splits, edges = [np.asarray(v).reshape(-1) for v in row[2:6]]
    assert all(len(v) == n for v in (ranges, times, splits, edges)), measurement_id
    assert set(np.unique(splits)).issubset({1, 2, 3}), measurement_id
    assert set(np.unique(edges)).issubset({0, 1}), measurement_id
    rows.append({'measurement_id': measurement_id, 'label': label, 'segments': n,
                 'nonfinite_signal_values': int((~np.isfinite(signal)).sum()),
                 'nonfinite_ranges': int((~np.isfinite(ranges)).sum()),
                 'nonfinite_times': int((~np.isfinite(times)).sum()),
                 'negative_time_steps': int((np.diff(times) < 0).sum()),
                 'edge_segments': int(edges.sum()),
                 'published_splits': tuple(np.unique(splits).astype(int))})
audit = pd.DataFrame(rows)
display(audit)
print('Observed segments:', audit.segments.sum(), 'Publisher reports: 75868')
display(audit.groupby('label').agg(measurements=('measurement_id','nunique'), segments=('segments','sum')))
shared = audit[audit.published_splits.map(len) > 1]
print('Measurements present in multiple published splits:', len(shared))
display(shared[['measurement_id','label','published_splits']])


In [ ]:
# Illustrative signal magnitude only: not a calibrated micro-Doppler feature.
label = audit.iloc[0]['label']
segment = np.asarray(data[0, 1])[:, 0].reshape(5, 256)
plt.figure(figsize=(10, 3))
plt.imshow(np.abs(segment), aspect='auto', origin='lower')
plt.xlabel('Azimuth sample index')
plt.ylabel('Range cell index')
plt.title(f'First segment, measurement 0: {label}')
plt.colorbar(label='Magnitude (native units)')
plt.tight_layout()
plt.show()


## Review before modeling
Record the observed counts and discrepancies. Inspect additional examples per class and edge status.
Determine whether a measurement-group split can preserve all intended classes. Do not randomly split correlated segments by default.
Agree on coarse class mapping and treatment of the reflector. Check session/subject overlap beyond the row IDs.
Do not claim synchronized visual verification, geographic localization, calibrated confidence, or continuous false-alert performance from this audit.
Save reviewed results only after executing against the verified real dataset; no results are prefilled here.
